# KD data generation — NLLB-200-1.3B teacher (resumable)
Beam-translates the 2M CCMatrix sources -> `kd_train.tsv` for sequence-level KD.
This is a **multi-session** job: it flushes progress every batch, so re-running
continues from where it stopped.

**Each session:** attach the `skripsi-edgenmten-id` dataset. To CONTINUE a prior
run, also attach *this notebook's previous output* (so `kd_train.tsv` +
`kd_train.progress.json` come back). GPU T4 + Internet On. **Save Version at the
end** so the partial `kd_train.tsv` persists for next time.


In [ ]:
import os
import sys
import glob
import shutil
import subprocess
import torch
import json as js

REPO='https://github.com/0wLzz/Edge-NMT.git'

if not os.path.isdir('Edge-NMT'):
    subprocess.run(['git','clone','--depth','1',REPO], check=True)

os.chdir('/kaggle/working/Edge-NMT'); sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers','sentencepiece','protobuf','sacremoses','accelerate'], check=True)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY (too slow)')

In [ ]:
# Assemble data/processed: source train.tsv (read-only symlink) + restore prior KD output
os.makedirs('data/processed/2M', exist_ok=True)
train_src = glob.glob('/kaggle/input/**/train.tsv', recursive=True)[0]
destination = 'data/processed/2M/train.tsv'

if os.path.islink(destination) or os.path.exists(destination):
    # Remove the existing symlink or file before creating a new symlink
    os.remove(destination)

os.symlink(train_src, destination)
print('train.tsv ->', train_src)

# Resume: if a previous kd_train.tsv is attached (this notebook's prior output), copy it back
prev = [p for p in glob.glob('/kaggle/input/**/kd_train.tsv', recursive=True)]

if prev:
    kd_dest = 'data/processed/kd_train.tsv'

    if os.path.islink(kd_dest) or os.path.exists(kd_dest):
        os.remove(kd_dest)

    shutil.copy(prev[0], kd_dest)

    progress_json = glob.glob('/kaggle/input/**/kd_train.progress.json', recursive=True)
    if progress_json: 
        shutil.copy(progress_json[0], 'data/processed/kd_train.progress.json')

    done = js.load(open('data/processed/kd_train.progress.json'))['processed_sources'] if progress_json else '?'
    print('RESUMING from prior output — already done:', done)
else:
    print('Fresh KD generation (no prior kd_train.tsv attached)')

In [ ]:
# Translate. Runs until the session ends; progress is flushed every batch, so a
# timeout is safe — just re-run next session (with the prior output attached).
# Smoke test first with a small --limit if you like.
subprocess.run([sys.executable,'-m',
                'model.training.generate_kd_dataset',
                '--config',
                'configs/config.yaml'], 
                check=True)

In [ ]:
# Progress + sample. SAVE VERSION after this so kd_train.tsv persists.
progress = 'data/processed/kd_train.progress.json'

if os.path.exists(progress):
    d = js.load(open(progress)); print('done %s / %s sources' % (d['processed_sources'], d['total_sources']))

print('--- Sample kd_train.tsv ---')
print(subprocess.run(['head','-3','data/processed/kd_train.tsv'],capture_output=True,text=True).stdout)